# DominusUltra independent GPU verification

This notebook checks out the full commit SHA shown in the setup cell, verifies that exact revision before installing dependencies, records the GPU/software environment, and produces correctness-gated JSON and Markdown reports. The target does not follow a moving branch. A benchmark number is valid only when the final verdict is **PASS**.

In [ ]:
import os
import re
import subprocess
import sys


def checkout_commit(repo_url, repo_dir, target_commit):
    if not re.fullmatch(r"[0-9a-f]{40}", target_commit):
        raise ValueError("Use the full, lowercase 40-character commit SHA")
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "clone", "--no-checkout", repo_url, repo_dir], check=True)
    else:
        status = subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=repo_dir, text=True
        ).strip()
        if status:
            raise RuntimeError(f"Save local changes before rerunning setup:\n{status}")
    subprocess.run(["git", "fetch", "origin", target_commit], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", "--detach", target_commit], cwd=repo_dir, check=True)
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True
    ).strip()
    if commit != target_commit:
        raise RuntimeError(f"Commit mismatch: expected {target_commit}, found {commit}")
    status = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo_dir, text=True
    ).strip()
    if status:
        raise RuntimeError(f"Expected a clean checkout, found:\n{status}")
    return commit

repo_url = "https://github.com/MiMindMendinc/DominusUltra.git"
target_commit = "a0d11750a9d5dfe858b2fa33348f8085e9bd5f2a"
repo_dir = "/content/DominusUltra"
commit = checkout_commit(repo_url, repo_dir, target_commit)
print("Target commit:", target_commit)
print("Verified commit:", commit)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
status = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
if status:
    raise RuntimeError(f"Installation changed the checkout:\n{status}")

In [ ]:
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
subprocess.run([
    sys.executable,
    "gpu_evidence.py",
    "--suite", "quick",
    "--dtype", "auto",
    "--warmup", "10",
    "--iterations", "50",
], check=True)

## Download and submit

The next cell downloads a ZIP containing the raw JSON and Markdown report. Attach both files to a new [Benchmark result issue](https://github.com/MiMindMendinc/DominusUltra/issues/new?template=benchmark_result.md). Failed runs are useful too—please include them unchanged.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/dominus-ultra-evidence", "zip", "benchmark_results")
files.download(archive)